# hai07 — training GPT 100M di Kaggle GPU
Input yang dibutuhkan (Add Input): dataset **hai07-prod-data**, **hai07-code**, dan (sesi ke-2 dst) **hai07-ckpt** (output sesi sebelumnya).
Settings: Accelerator = GPU T4 x2, Persistence = Files. Set `SMOKE=True` dulu, lalu `False` untuk run penuh.

In [ ]:
SMOKE = True            # True: 20 step uji kecepatan; False: training penuh
ENV = "production_100m"
MAX_STEPS = 25600       # 2 epoch (~14.390 step/epoch), ~17 jam di T4. Jangan diubah antar sesi (jadwal LR cosine bergantung nilai ini).
EPOCHS = 2
TIME_LIMIT_S = 11*3600 + 1200   # berhenti rapi < 12 jam; sesi berikutnya resume
SAVE_EVERY = 250

In [ ]:
import os, glob, shutil, subprocess, sys, time, threading, zipfile
!nvidia-smi
WORK = "/kaggle/working/hai07-ocd"
os.makedirs(WORK, exist_ok=True)

# Diagnostik: tampilkan isi /kaggle/input (maks 3 tingkat)
for root, dirs, files in os.walk("/kaggle/input"):
    if root.count(os.sep) - 2 >= 3: dirs[:] = []; continue
    print(root, "->", (dirs + files)[:8])

def find_dir(name):
    for root, dirs, _ in os.walk("/kaggle/input"):
        if name in dirs and os.path.exists(os.path.join(root, name, "__pycache__")) is not None:
            return os.path.join(root, name)
    return None

# Jika kode masih berupa zip (tidak terekstrak otomatis), ekstrak dulu
if find_dir("apps") is None:
    zips = glob.glob("/kaggle/input/**/hai07-code*.zip", recursive=True)
    assert zips, "apps/ dan hai07-code.zip tidak ditemukan; periksa isi dataset hai07-code (lihat daftar di atas)"
    zipfile.ZipFile(zips[0]).extractall("/kaggle/working/code_src")
    SRC_ROOT = "/kaggle/working/code_src"
else:
    SRC_ROOT = os.path.dirname(find_dir("apps"))

for d in ("apps", "tools"):
    src = os.path.join(SRC_ROOT, d)
    assert os.path.isdir(src), f"folder {d} tidak ada di {SRC_ROOT}"
    shutil.rmtree(f"{WORK}/{d}", ignore_errors=True)
    shutil.copytree(src, f"{WORK}/{d}")

data_dir = os.path.dirname(glob.glob("/kaggle/input/**/train_tokens.bin", recursive=True)[0])
# /kaggle/input adalah network storage (lambat untuk akses acak memmap) -> salin ke disk lokal sementara
LOCAL = "/kaggle/temp/prod" if os.path.isdir("/kaggle/temp") else "/tmp/prod"
if not os.path.exists(LOCAL):
    t_cp = time.time(); shutil.copytree(data_dir, LOCAL)
    for f in glob.glob(LOCAL + "/*.bin") + glob.glob(LOCAL + "/*.npy"):
        with open(f, "rb") as fh:            # pemanasan page cache
            while fh.read(64 << 20): pass
    print(f"data disalin ke {LOCAL} dalam {time.time()-t_cp:.0f}s")
os.makedirs(f"{WORK}/data", exist_ok=True)
if os.path.islink(f"{WORK}/data/prod"): os.remove(f"{WORK}/data/prod")
os.symlink(LOCAL, f"{WORK}/data/prod")
!df -h /kaggle/temp /tmp /kaggle/working | cat
print("kode dari:", SRC_ROOT, "| data:", data_dir)
os.chdir(WORK)
import torch; print(torch.__version__, torch.cuda.get_device_name(0), "bf16:", torch.cuda.is_bf16_supported())

In [ ]:
# Pulihkan checkpoint dari sesi sebelumnya (jika ada)
CK = f"{WORK}/reports/checkpoints_100m"
os.makedirs(CK, exist_ok=True)
prev = glob.glob("/kaggle/input/**/checkpoints_100m/gpt_*.pt", recursive=True) + glob.glob("/kaggle/input/**/gpt_step_*.pt", recursive=True) + glob.glob("/kaggle/input/**/gpt_last.pt", recursive=True) + glob.glob("/kaggle/input/**/gpt_best.pt", recursive=True)
for f in set(prev):
    if "hai07-prod-data" in f: continue
    dst = os.path.join(CK, os.path.basename(f))
    if not os.path.exists(dst):
        shutil.copy(f, dst); print("restored", dst)

import re
def step_of(f):
    m = re.search(r"gpt_step_(\d+)\.pt$", f)
    return int(m.group(1)) if m else (10**12 if f.endswith("gpt_last.pt") else -1)

def newest_resume():
    # pilih berdasarkan nomor step (bukan mtime, yang berubah saat file disalin)
    c = glob.glob(f"{CK}/gpt_step_*.pt") + glob.glob(f"{CK}/gpt_last.pt")
    return max(c, key=step_of) if c else None
print("resume dari:", newest_resume())

In [ ]:
def prune_loop(stop):
    # checkpoint ~1,2 GB; simpan hanya 2 terbaru agar tidak melewati kuota 20 GB
    while not stop.is_set():
        steps = sorted(glob.glob(f"{CK}/gpt_step_*.pt"), key=step_of)
        for f in steps[:-2]:
            try: os.remove(f)
            except OSError: pass
        stop.wait(60)

stop = threading.Event(); threading.Thread(target=prune_loop, args=(stop,), daemon=True).start()

cmd = ["python", "tools/train_gptV2.py", "--env", ENV, "--device", "cuda", "--amp",
       "--token-file", "data/prod/train_tokens.bin",
       "--val-token-file", "data/prod/validation_tokens.bin",
       "--scheduler", "cosine", "--num-workers", "2"]
if SMOKE:
    cmd += ["--max-steps", "60", "--log-every", "5", "--save-every", "1000", "--val-every", "1000", "--val-steps", "20"]
else:
    cmd += ["--epochs", str(EPOCHS), "--max-steps", str(MAX_STEPS), "--log-every", "50", "--save-every", str(SAVE_EVERY),
            "--val-every", "500", "--val-steps", "100"]
    r = newest_resume()
    if r: cmd += ["--resume", r]
print(" ".join(cmd))
t0 = time.time()
cmd[0:1] = ["python", "-u"]
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
def watchdog():
    if not SMOKE and not stop.wait(TIME_LIMIT_S):
        print("[notebook] batas waktu tercapai -> menghentikan; lanjutkan di sesi berikutnya (resume otomatis)", flush=True)
        p.terminate()
threading.Thread(target=watchdog, daemon=True).start()
for line in p.stdout:          # tampilkan log training langsung di cell ini
    print(line, end="", flush=True)
p.wait()
stop.set()
print("elapsed(h):", (time.time()-t0)/3600, "returncode:", p.returncode)


In [ ]:
# Ringkasan output yang akan tersimpan
!ls -la {CK}; du -sh /kaggle/working